# Publish the calibrated students (targets F and E)
Packages three seeds of each student with their dials, checks the package reproduces the training notebook, measures the published files, writes the model cards, and uploads. Dry run first (`PUBLISH = False`).

## Cell 1 — Install, Drive and Hugging Face login
A GPU runtime is recommended (Cells 6 and 7 run six students). The login needs a WRITE token.

In [ ]:
!pip install -q "transformers>=4.48" datasets safetensors huggingface_hub
import os, sys, json, pickle, random, shutil, hashlib, importlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from google.colab import drive
drive.mount("/content/drive")
from huggingface_hub import notebook_login
notebook_login()

## Cell 2 — Settings
What to publish, where the students are on Drive, and the report's numbers to compare against.

In [ ]:
PUBLISH  = True                                   # True: create the two model repositories and upload
BASE_DIR = "/content/drive/MyDrive/ctd_student_v2"
EXPORT   = "/content/ctd_export"
DATASET  = "m-lagnajit/calibrated-typed-decisions"
REPOS    = {"F": "m-lagnajit/ctd-student-f", "E": "m-lagnajit/ctd-student-e"}
SEEDS    = ["seed0", "seed1", "seed2"]
# seed 0: the original students; seeds 1 and 2: the retrained ones from Step 7 (the only ones with saved weights)
WEIGHTS  = {"seed0": "{base}/student_{run}.pt", "seed1": "{base}/seed1_rerun/student_{run}.pt",
            "seed2": "{base}/seed2_rerun/student_{run}.pt"}
PREDS    = {k: v.replace("student_", "preds_").replace(".pt", ".pkl") for k, v in WEIGHTS.items()}
CONFIG   = dict(base_model="answerdotai/ModernBERT-base", max_len=512, max_question_tokens=96, max_option_tokens=32,
                seeds=SEEDS, dataset=DATASET, dataset_revision="main (v2.1)")
SNIPS_FINGERPRINT_EXPECTED = "b0096e8b4312"        # the SNIPS question set used in the experiments

# Dials: the same search grids and minimum group size as the training notebook (Cells 4 and 16)
GRID, B_GRID, MIN_FIT_ROWS, N_BINS = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200)), np.linspace(-8.0, 8.0, 321), 30, 10

# The report's ensemble results (Appendix B), for comparison with the published files
REPORT = {"F": {"test": (0.928, 0.006, 0.225), "snips": (0.958, 0.020, 0.126), "heldout": (0.595, 0.051, 0.899)},
          "E": {"test": (0.918, 0.013, 0.241), "snips": (0.965, 0.020, 0.114), "heldout": (0.587, 0.081, 0.926)}}
LINKS = {"report": "https://lagnajitmoharana.web.app/calibrated-typed-decisions-technical-report.pdf",
         "blog": "https://lagnajitmoharana.web.app/how-sure-is-sure-understanding-llm-calibration",
         "code": "https://github.com/Lagnajit09/100x_AI_ML/tree/main/calibrated-typed-decision",
         "dataset": f"https://huggingface.co/datasets/{DATASET}"}

## Cell 3 — The package file
Exactly what users download. Written here, then copied into each model folder.

In [ ]:
%%writefile /content/ctd_student.py
"""Calibrated typed-decision students, trained on m-lagnajit/calibrated-typed-decisions.

A student reads a text, a question and a list of options in one pass, and returns a calibrated chance for every
option. Each repository holds three students trained with different seeds; their dialed chances are averaged.

    student = CalibratedStudent.from_pretrained("m-lagnajit/ctd-student-f")
    student.predict_one(instructions="Which banking intent best describes this customer message?",
                        text="I still haven't received my new card, what should I do?",
                        options=["card arrival", "card delivery estimate", "lost or stolen card"])

The input layout, token limits and dials reproduce the training notebook exactly:
    [CLS] question [SEP] [MASK] option 1 [MASK] option 2 ... [SEP] text [SEP]
"""
import json
import os

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

QUESTION_TYPES = {"choice": "choice", "yes/no": "noul", "noul": "noul"}


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def option_texts(row):
    """What the student reads for each option, in stored order: 'label: description'; yes/no as 'no' and 'yes'."""
    if row["type"] == "noul":
        return ["no", "yes"]
    descs = row.get("option_descriptions") or [""] * len(row["options"])
    return [f"{o}: {d}" if d else o for o, d in zip(row["options"], descs)]


def apply_dial(scores, qtype, dial):
    """A temperature for every question; for yes/no, also a shift added to the 'yes' score."""
    T, b = dial
    z = np.asarray(scores, dtype=float) / T
    if qtype == "noul":
        z[1] += b
    return softmax(z)


class _Student(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.body = AutoModel.from_config(config)
        self.scorer = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, marker_pos, option_mask):
        h = self.body(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state   # (B, L, H)
        idx = marker_pos.unsqueeze(-1).expand(-1, -1, h.size(-1))                             # (B, K, H)
        scores = self.scorer(torch.gather(h, 1, idx)).squeeze(-1)                             # (B, K)
        return scores.masked_fill(~option_mask, float("-inf"))                                # empty slots


class CalibratedStudent:
    """Three seeds of one student, with their dials. Use from_pretrained(repo_id or local folder)."""

    def __init__(self, folder, device=None):
        from safetensors.torch import load_file
        with open(os.path.join(folder, "config.json")) as f:
            self.config = json.load(f)
        with open(os.path.join(folder, "dials.json")) as f:
            self.dials = {seed: {t: tuple(d) for t, d in v.items()} for seed, v in json.load(f).items()}
        c = self.config
        self.max_len, self.max_q, self.max_opt = c["max_len"], c["max_question_tokens"], c["max_option_tokens"]
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(c["base_model"])
        self.CLS, self.SEP = self.tok.cls_token_id, self.tok.sep_token_id
        self.MASK, self.PAD = self.tok.mask_token_id, self.tok.pad_token_id
        model_config = AutoConfig.from_pretrained(c["base_model"])
        if hasattr(model_config, "reference_compile"):
            model_config.reference_compile = False
        self.models = {}
        for seed in c["seeds"]:
            m = _Student(model_config)
            state = load_file(os.path.join(folder, f"{seed}.safetensors"))
            m.load_state_dict({k: v.float() for k, v in state.items()})     # stored in 16-bit, run in 32-bit
            self.models[seed] = m.float().to(self.device).eval()

    @classmethod
    def from_pretrained(cls, repo_or_folder, device=None):
        if os.path.isdir(repo_or_folder):
            return cls(repo_or_folder, device)
        from huggingface_hub import snapshot_download
        return cls(snapshot_download(repo_or_folder), device)

    # ---------- input, exactly as in training ----------
    def _tokenize(self, row):
        enc = lambda s: self.tok(s, add_special_tokens=False)["input_ids"]
        q = enc(row["instructions"])[:self.max_q]
        opts = [enc(" " + o)[:self.max_opt] for o in option_texts(row)]
        txt = enc(" " + row["state"])
        ids = [self.CLS] + q + [self.SEP]
        marker_pos = []
        for o in opts:                                  # stored order: no shuffling at prediction time
            marker_pos.append(len(ids))
            ids += [self.MASK] + o
        ids += [self.SEP]
        room = self.max_len - len(ids) - 1              # only the text can ever be cut
        return ids + txt[:room] + [self.SEP], marker_pos

    def _batch(self, rows):
        items = [self._tokenize(r) for r in rows]
        B, L, K = len(rows), max(len(i) for i, _ in items), max(len(m) for _, m in items)
        input_ids = torch.full((B, L), self.PAD, dtype=torch.long)
        attn = torch.zeros((B, L), dtype=torch.long)
        marker_pos = torch.zeros((B, K), dtype=torch.long)          # empty slots point at position 0
        option_mask = torch.zeros((B, K), dtype=torch.bool)
        for b, (ids, mp) in enumerate(items):
            input_ids[b, :len(ids)] = torch.tensor(ids)
            attn[b, :len(ids)] = 1
            marker_pos[b, :len(mp)] = torch.tensor(mp)
            option_mask[b, :len(mp)] = True
        return [t.to(self.device) for t in (input_ids, attn, marker_pos, option_mask)], [len(m) for _, m in items]

    @torch.no_grad()
    def raw_scores(self, rows, seed, batch_size=64):
        """Undialed scores of one seed, one array per row, in the options' stored order."""
        model, out = self.models[seed], []
        for i in range(0, len(rows), batch_size):
            feed, ks = self._batch(rows[i:i + batch_size])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.device == "cuda"):
                s = model(*feed).float().cpu().numpy()
            out += [s[b, :k] for b, k in enumerate(ks)]
        return out

    # ---------- predictions ----------
    def predict(self, rows, batch_size=64, return_seeds=False):
        """Calibrated chances for each row (a dict with 'instructions', 'state', 'options', 'type' and optionally
        'option_descriptions'): each seed's scores pass through its own dial, then the seeds are averaged."""
        rows = [dict(r, type=QUESTION_TYPES[r.get("type", "choice")]) for r in rows]
        for r in rows:
            if r["type"] == "noul":
                r["options"] = ["false", "true"]
        per_seed = {seed: [apply_dial(s, r["type"], self.dials[seed][r["type"]])
                           for s, r in zip(self.raw_scores(rows, seed, batch_size), rows)] for seed in self.models}
        ensemble = [np.mean([per_seed[seed][i] for seed in self.models], axis=0) for i in range(len(rows))]
        return (ensemble, per_seed) if return_seeds else ensemble

    def predict_one(self, instructions, text, options=None, descriptions=None, question_type="choice"):
        """One question. Choice: pass the options (and, ideally, a short description of each).
        Yes/no: put the candidate label in the question, and leave options empty."""
        row = {"instructions": instructions, "state": text, "type": question_type,
               "options": list(options or []), "option_descriptions": list(descriptions or [])}
        chances = self.predict([row])[0]
        names = ["no", "yes"] if QUESTION_TYPES[question_type] == "noul" else row["options"]
        return dict(zip(names, (round(float(p), 4) for p in chances)))

Writing /content/ctd_student.py


## Cell 4 — Collect and repackage the students
Three seeds per target, converted to 16-bit safetensors, plus config.json.

In [ ]:
from safetensors.torch import save_file
sys.path.insert(0, "/content")
for run in REPOS:
    out = f"{EXPORT}/{run}"
    os.makedirs(out, exist_ok=True)
    for seed in SEEDS:
        path = WEIGHTS[seed].format(base=BASE_DIR, run=run)
        assert os.path.exists(path), f"missing weights: {path}"
        state = torch.load(path, map_location="cpu")
        assert any(k.startswith("body.") for k in state) and any(k.startswith("scorer.") for k in state), \
            f"{path} doesn't look like a full student (body + scorer)"
        state = {k: (v.to(torch.float16) if v.is_floating_point() else v).contiguous() for k, v in state.items()}
        save_file(state, f"{out}/{seed}.safetensors")
    with open(f"{out}/config.json", "w") as f:
        json.dump(dict(CONFIG, target=run), f, indent=2)
    with open(f"{out}/dials.json", "w") as f:                    # placeholder until Cell 6 fits the real dials
        json.dump({s: {"choice": [1.0, 0.0], "noul": [1.0, 0.0]} for s in SEEDS}, f)
    shutil.copy("/content/ctd_student.py", out)
    sizes = {f: f"{os.path.getsize(f'{out}/{f}') / 1e6:.0f} MB" for f in sorted(os.listdir(out))}
    print(run, sizes)

F {'config.json': '0 MB', 'ctd_student.py': '0 MB', 'dials.json': '0 MB', 'seed0.safetensors': '298 MB', 'seed1.safetensors': '298 MB', 'seed2.safetensors': '298 MB'}
E {'config.json': '0 MB', 'ctd_student.py': '0 MB', 'dials.json': '0 MB', 'seed0.safetensors': '298 MB', 'seed1.safetensors': '298 MB', 'seed2.safetensors': '298 MB'}


## Cell 5 — The evaluation data
Dataset v2.1, and the SNIPS questions rebuilt with the exact code of the experiments (Cell 26).

In [ ]:
ds = load_dataset(DATASET, revision="main")
DATA = {split: list(ds[split]) for split in ["train", "validation", "test", "heldout"]}
print({split: len(rows) for split, rows in DATA.items()})
SEED = 0
pretokenize = lambda rows: None                 # the package tokenizes by itself

# ---- the training notebook's Cell 26, unchanged ----
# Step 6/7: BUILD the SNIPS questions only. Needs: Cells 1-7.
# Run it BEFORE training (Cells 14-15) in Step 7, so each new student also answers SNIPS.
import zipfile, glob, re, urllib.request
from collections import Counter
SNIPS_URL        = "https://github.com/sonos/nlu-benchmark/archive/refs/heads/master.zip"   # authors' repo, CC0-1.0
SNIPS_DIR        = "/content/snips"
SNIPS_PER_INTENT = 150
SNIPS_DESC = {   # written by us, for the student, like the v2 descriptions
    "add to playlist":        "add a song or artist to a playlist",
    "book restaurant":        "reserve a table at a restaurant",
    "get weather":            "ask about the weather or the forecast",
    "play music":             "play a song, album, artist, or playlist",
    "rate book":              "give a rating to a book or other written work",
    "search creative work":   "find a book, song, film, show, or game",
    "search screening event": "find showtimes, or where a film is playing",
}
SNIPS_SOURCE = "sonos/nlu-benchmark (SNIPS 2017)"
SNIPS_SEED   = 0          # fixed on purpose: NOT the training SEED, so every student sees identical questions

# 1. Download and read the authors' files
os.makedirs(SNIPS_DIR, exist_ok=True)
zpath = f"{SNIPS_DIR}/nlu-benchmark.zip"
if not os.path.exists(zpath):
    urllib.request.urlretrieve(SNIPS_URL, zpath)
with zipfile.ZipFile(zpath) as z:
    z.extractall(SNIPS_DIR)

def read_json_any(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8", "latin-1"):           # a few of these files aren't valid UTF-8
        try:
            return json.loads(raw.decode(enc))
        except (UnicodeDecodeError, json.JSONDecodeError):
            pass
    raise ValueError(f"can't read {path}")

def humanize_intent(name):                     # "SearchScreeningEvent" -> "search screening event"
    return " ".join(w.lower() for w in re.findall(r"[A-Z][a-z]*", name))

def utterances(path):
    d = read_json_any(path)
    (intent, items), = d.items()
    return humanize_intent(intent), [" ".join("".join(seg["text"] for seg in it["data"]).split()) for it in items]

base = glob.glob(f"{SNIPS_DIR}/**/2017-06-custom-intent-engines", recursive=True)
assert len(base) == 1, f"couldn't find the 2017 folder: {base}"
POOL, SEEN = {}, {}
for kind in ("validate_*.json", "train_*_full.json"):       # validation files first, then the training files
    for path in sorted(glob.glob(f"{base[0]}/*/{kind}")):
        intent, texts = utterances(path)
        POOL.setdefault(intent, []); SEEN.setdefault(intent, set())
        for t in texts:
            if t and t.lower() not in SEEN[intent]:
                POOL[intent].append(t); SEEN[intent].add(t.lower())
SNIPS_NAMES = sorted(POOL)
print("intents:", SNIPS_NAMES)
assert set(SNIPS_NAMES) == set(SNIPS_DESC), f"unexpected intents: {SNIPS_NAMES}"
assert all(len(v) >= SNIPS_PER_INTENT for v in POOL.values()), {k: len(v) for k, v in POOL.items()}

# 2. Overlap check: is any SNIPS request word-for-word in the training data?
train_texts = {r["state"].strip().lower() for r in DATA["train"]}
picked = {k: v[:SNIPS_PER_INTENT] for k, v in POOL.items()}
overlap = sum(t.lower() in train_texts for v in picked.values() for t in v)
print(f"SNIPS requests found word-for-word in the training data: {overlap} of {sum(map(len, picked.values()))}")

# 3. Questions, built like the v2 questions (descriptions, 3-7 options, 30% yes/no), seeded
rng = random.Random(f"{SNIPS_SEED}-snips")
SNIPS = []
for gi, intent in enumerate(SNIPS_NAMES):
    for j, text in enumerate(picked[intent]):
        others = [k for k in range(len(SNIPS_NAMES)) if k != gi]
        if rng.random() < 0.3:
            is_true = rng.random() < 0.5
            cand = gi if is_true else rng.choice(others)
            label = SNIPS_NAMES[cand]
            q = dict(type="noul", options=["false", "true"], option_descriptions=["", ""], candidate=label,
                     instructions=f"Is this request to a voice assistant asking for: {label} ({SNIPS_DESC[label]})?",
                     gold_index=int(is_true))
        else:
            k = rng.randint(3, len(SNIPS_NAMES))
            opts = [gi] + rng.sample(others, k - 1)
            rng.shuffle(opts)
            names = [SNIPS_NAMES[o] for o in opts]
            q = dict(type="choice", options=names, option_descriptions=[SNIPS_DESC[n] for n in names], candidate="",
                     instructions="Which intent best describes this request to a voice assistant?",
                     gold_index=opts.index(gi))
        SNIPS.append(dict(id=f"snips-{gi}-{j:04d}", source=SNIPS_SOURCE, state=text,
                          gold=q["options"][q["gold_index"]], **q))
pretokenize(SNIPS)
print(f"{len(SNIPS)} questions:", dict(Counter(r["type"] for r in SNIPS)))
import hashlib
SNIPS_FINGERPRINT = hashlib.md5(json.dumps([(r["id"], r["options"], r["gold_index"], r["state"]) for r in SNIPS])
                                .encode()).hexdigest()[:12]
print("question-set fingerprint:", SNIPS_FINGERPRINT, "(must be the same in every session)")
assert SNIPS_FINGERPRINT == SNIPS_FINGERPRINT_EXPECTED, \
    f"SNIPS questions differ from the experiments' ({SNIPS_FINGERPRINT} vs {SNIPS_FINGERPRINT_EXPECTED})"
print("SNIPS questions identical to the experiments'")
SPLITS = {"test": DATA["test"], "snips": SNIPS, "heldout": DATA["heldout"]}

README.md:   0%|          | 0.00/20.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.15MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  889kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  898kB            

... (4 outputs left out)

Generating validation split:   0%|          | 0/1799 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1797 [00:00<?, ? examples/s]

Generating heldout split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'train': 16995, 'validation': 1799, 'test': 1797, 'heldout': 1000}
intents: ['add to playlist', 'book restaurant', 'get weather', 'play music', 'rate book', 'search creative work', 'search screening event']
SNIPS requests found word-for-word in the training data: 0 of 1050
1050 questions: {'choice': 734, 'noul': 316}
question-set fingerprint: b0096e8b4312 (must be the same in every session)
SNIPS questions identical to the experiments'


## Cell 6 — Taste test 1, then the dials
Each student, loaded through the package, must reproduce its saved predictions. Then its dials are fitted.

In [ ]:
import ctd_student
importlib.reload(ctd_student)

def fit_one(scores, rows):                       # the training notebook's dial fitting (Cell 24), unchanged
    y = np.array([r["gold_index"] for r in rows])
    if rows[0]["type"] == "noul":
        gap = np.array([s[1] - s[0] for s in scores])
        with np.errstate(over="ignore"):
            z = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]
            p = 1.0 / (1.0 + np.exp(-z))
        pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
        i, j = np.unravel_index(np.argmin(pen), pen.shape)
        return float(GRID[i]), float(B_GRID[j])
    total, by_k = np.zeros(len(GRID)), {}
    for s, g in zip(scores, y):
        by_k.setdefault(len(s), []).append((s, g))
    for items in by_k.values():
        S = np.array([s for s, _ in items], dtype=float)
        g = np.array([g for _, g in items])
        Z = S[None] / GRID[:, None, None]
        Z = Z - Z.max(axis=2, keepdims=True)
        logp = Z - np.log(np.exp(Z).sum(axis=2, keepdims=True))
        total += -logp[:, np.arange(len(g)), g].sum(axis=1)
    return float(GRID[int(np.argmin(total))]), 0.0

def two_dials(scores, rows):
    pooled = {}
    for qtype in ("choice", "noul"):
        idx = [i for i, r in enumerate(rows) if r["type"] == qtype]
        pooled[qtype] = (fit_one([scores[i] for i in idx], [rows[i] for i in idx])
                         if len(idx) >= MIN_FIT_ROWS else (1.0, 0.0))
    return pooled

CHECK_N, DIALS = 64, {}
for run in REPOS:
    student = ctd_student.CalibratedStudent.from_pretrained(f"{EXPORT}/{run}")
    DIALS[run] = {}
    for seed in SEEDS:
        with open(PREDS[seed].format(base=BASE_DIR, run=run), "rb") as f:
            saved = pickle.load(f)["test"][:CHECK_N]
        again = student.raw_scores(DATA["test"][:CHECK_N], seed)
        gap = max(float(np.max(np.abs(np.asarray(a) - np.asarray(b)))) for a, b in zip(again, saved))
        same = sum(int(np.argmax(a)) == int(np.argmax(b)) for a, b in zip(again, saved))
        print(f"{run} {seed}: largest gap from its saved predictions {gap:.4f}; same top pick {same}/{CHECK_N}")
        assert gap < 0.05 and same == CHECK_N, f"{run} {seed}: the package doesn't reproduce the training notebook"
        DIALS[run][seed] = two_dials(student.raw_scores(DATA["validation"], seed), DATA["validation"])
    with open(f"{EXPORT}/{run}/dials.json", "w") as f:
        json.dump({s: {t: list(d) for t, d in v.items()} for s, v in DIALS[run].items()}, f, indent=2)
    print(run, "dials:", {s: {t: (round(d[0], 3), round(d[1], 2)) for t, d in v.items()} for s, v in DIALS[run].items()})
    del student; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

F seed0: largest gap from its saved predictions 0.0127; same top pick 64/64
F seed1: largest gap from its saved predictions 0.0137; same top pick 64/64
F seed2: largest gap from its saved predictions 0.0156; same top pick 64/64
F dials: {'seed0': {'choice': (0.786, 0.0), 'noul': (0.821, 0.2)}, 'seed1': {'choice': (0.821, 0.0), 'noul': (0.803, 0.9)}, 'seed2': {'choice': (0.803, 0.0), 'noul': (0.858, 0.5)}}
E seed0: largest gap from its saved predictions 0.0088; same top pick 64/64
E seed1: largest gap from its saved predictions 0.0127; same top pick 64/64
E seed2: largest gap from its saved predictions 0.0117; same top pick 64/64
E dials: {'seed0': {'choice': (0.644, 0.0), 'noul': (0.63, 0.65)}, 'seed1': {'choice': (0.704, 0.0), 'noul': (0.673, 0.9)}, 'seed2': {'choice': (0.704, 0.0), 'noul': (0.735, 0.55)}}


## Cell 7 — Taste test 2: measure the published students
Through the public predict() function, on the three tests; compared with the report.

In [ ]:
def per_row(probs_list, rows):                   # the training notebook's measures (Cell 13), unchanged
    conf    = np.array([p.max() for p in probs_list])
    correct = np.array([float(p.argmax() == r["gold_index"]) for p, r in zip(probs_list, rows)])
    nll     = np.array([-np.log(max(p[r["gold_index"]], 1e-12)) for p, r in zip(probs_list, rows)])
    return conf, correct, nll

def ece_from(conf, correct):
    bins  = np.minimum((conf * N_BINS).astype(int), N_BINS - 1)
    said  = np.bincount(bins, weights=conf,    minlength=N_BINS)
    right = np.bincount(bins, weights=correct, minlength=N_BINS)
    return float(np.abs(right - said).sum() / len(conf))

def metrics(probs_list, rows):
    conf, correct, nll = per_row(probs_list, rows)
    return {"accuracy": correct.mean(), "ECE": ece_from(conf, correct), "log loss": nll.mean()}

RESULTS, rows_out = {}, []
for run in REPOS:
    student = ctd_student.CalibratedStudent.from_pretrained(f"{EXPORT}/{run}")      # with its real dials now
    RESULTS[run] = {}
    for split, rows in SPLITS.items():
        ens, per_seed = student.predict(rows, return_seeds=True)
        singles = [metrics(per_seed[s], rows) for s in SEEDS]
        e = metrics(ens, rows)
        assert e["log loss"] <= np.mean([m["log loss"] for m in singles]) + 1e-12, "Jensen check failed"
        RESULTS[run][split] = {"ensemble": e, "singles": singles}
        rep = REPORT[run][split]
        rows_out.append({"student": run, "test": split, "accuracy": e["accuracy"], "ECE": e["ECE"], "log loss": e["log loss"],
                         "report accuracy": rep[0], "report ECE": rep[1], "report log loss": rep[2],
                         "log loss change": e["log loss"] - rep[2]})
    del student; torch.cuda.empty_cache()
COMPARE = pd.DataFrame(rows_out)
print("ENSEMBLES OF THE PUBLISHED FILES vs THE REPORT")
display(COMPARE.round(3))
print("Expect SNIPS to match closely (same students). Test and PubMedQA can differ a little: the report's ensembles used "
      "the original seeds 1 and 2, whose weights weren't saved.")
COMPARE.to_csv(f"{BASE_DIR}/published_students_results.csv", index=False)

ENSEMBLES OF THE PUBLISHED FILES vs THE REPORT


,student,test,accuracy,ECE,log loss,report accuracy,report ECE,report log loss,log loss change
0,F,test,0.929,0.015,0.223,0.928,0.006,0.225,-0.002
1,F,snips,0.958,0.020,0.126,0.958,0.020,0.126,0.000
2,F,heldout,0.608,0.085,0.910,0.595,0.051,0.899,0.011
3,E,test,0.920,0.014,0.237,0.918,0.013,0.241,-0.004
4,E,snips,0.965,0.020,0.114,0.965,0.020,0.114,0.000
5,E,heldout,0.607,0.097,0.918,0.587,0.081,0.926,-0.008


Expect SNIPS to match closely (same students). Test and PubMedQA can differ a little: the report's ensembles used the original seeds 1 and 2, whose weights weren't saved.


## Cell 8 — The model cards
Generated from the numbers measured in Cell 7.

In [ ]:
NAMES = {"test": "familiar data (test split)", "snips": "a new label set (SNIPS)", "heldout": "a new kind of judgment (PubMedQA)"}
ABOUT = {
    "F": dict(target="`0.5 × one-hot(gold) + 0.5 × corrected_soft_label`",
              use="**data like the six training sources**: short texts and a fixed set of options, such as intents, "
                  "emotions, review sentiment or citation purposes",
              why="it had the best familiar-data accuracy and calibration in the report, and was never reliably "
                  "beaten there"),
    "E": dict(target="`corrected_soft_label` (the teacher's opinion, with its wrong top picks corrected)",
              use="**a new label set of the same kind**: decisions like the training sources, but with categories "
                  "the student has never seen",
              why="on SNIPS (7 unseen intents) it beat every other target on at least one measure, and nothing "
                  "beat it"),
}
def mr(vals):
    vals = np.array(vals); return f"{vals.mean():.3f} ({vals.min():.3f}–{vals.max():.3f})"

def card(run):
    a, R, repo = ABOUT[run], RESULTS[run], REPOS[run]
    table = "\n".join(
        f"| {NAMES[sp]} | {R[sp]['ensemble']['accuracy']:.3f} | {R[sp]['ensemble']['ECE']:.3f} | "
        f"{R[sp]['ensemble']['log loss']:.3f} | {mr([m['log loss'] for m in R[sp]['singles']])} |"
        for sp in SPLITS)
    return f"""---
license: apache-2.0
language:
- en
base_model: answerdotai/ModernBERT-base
datasets:
- {DATASET}
pipeline_tag: text-classification
tags:
- calibration
- typed-decisions
- knowledge-distillation
- modernbert
---

# ctd-student-{run.lower()}: calibrated typed decisions, target {run}

A small (about 150M parameters) classifier that reads a **text, a question and a list of options** in one pass,
and returns a **calibrated chance for every option**: when it says 80%, it should be right about 80% of the time.
This repository holds **three students** trained with different seeds; their chances are averaged.

**Use it for {a['use']}.** It was chosen because {a['why']}.
It is **not** reliable for new *kinds* of judgment: on PubMedQA, the best students in the report were honest
about being unsure, but only slightly better than knowing how often each answer occurs.

Details: [technical report]({LINKS['report']}) · [blog post]({LINKS['blog']}) · [dataset]({LINKS['dataset']}) ·
[code]({LINKS['code']}). For new label sets, see [ctd-student-e](https://huggingface.co/{REPOS['E']}); for data like
the training sources, see [ctd-student-f](https://huggingface.co/{REPOS['F']}).

## Usage

```python
# pip install "transformers>=4.48" safetensors huggingface_hub
import importlib.util
from huggingface_hub import hf_hub_download

path = hf_hub_download("{repo}", "ctd_student.py")
spec = importlib.util.spec_from_file_location("ctd_student", path)
ctd_student = importlib.util.module_from_spec(spec); spec.loader.exec_module(ctd_student)

student = ctd_student.CalibratedStudent.from_pretrained("{repo}")

# A choice question: options, ideally with a short description each
student.predict_one(
    instructions="Which banking intent best describes this customer message?",
    text="I still haven't received my new card, what should I do?",
    options=["card arrival", "card delivery estimate", "lost or stolen card", "activate my card"],
    descriptions=["asking where an ordered card is", "asking how long delivery takes",
                  "reporting a lost or stolen card", "asking how to activate a card"])

# A yes/no question: put the candidate label in the question
student.predict_one(
    instructions="Is the customer's intent in this message: card arrival (asking where an ordered card is)?",
    text="I still haven't received my new card, what should I do?", question_type="yes/no")
```

`predict_one` returns a dictionary of chances. For many questions at once, use `student.predict(rows)` with dicts
holding `instructions`, `state`, `options`, `option_descriptions` and `type` (`"choice"` or `"yes/no"`).

## Results

Measured **on the files in this repository** (three-seed ensemble, each seed with its own dials fitted on the
dataset's validation split). Last column: single students, mean (lowest–highest).

| Test | Accuracy | ECE | Log loss | Log loss, single students |
|---|---|---|---|---|
{table}

For comparison: on PubMedQA, always answering "yes" scores accuracy 0.552, and always predicting the label
frequencies scores log loss 0.937. These numbers can differ slightly from the report's ensembles: the report used the
original seeds 1 and 2, whose weights were not saved, while this repository contains the same recipe retrained
with the same seeds (SNIPS results in the report already used these students).

## Training

- **Student:** ModernBERT-base with one shared linear scorer over a `[MASK]` marker before each option.
- **Target:** {a['target']}, on the 16,995 training questions of [{DATASET}]({LINKS['dataset']}).
- **Recipe:** 3 epochs, batch 16, AdamW (weight decay 0.01), learning rates 3e-5 (body) and 1e-3 (scorer), 10%
  warmup, gradient clipping 1.0, mixed precision; the epoch with the lowest validation log loss is kept.
- **Dials:** one temperature for choice questions, and one temperature plus a "yes" shift for yes/no questions,
  per seed, fitted on validation (`dials.json`).

## Limitations

- English only; up to 8 options; inputs longer than 512 tokens are cut (only the text, never the options).
- Calibrated on data like its training sources; on unfamiliar *kinds* of judgment it is honest but knows little.
- Its dials were fitted on familiar data; they may over- or under-correct on very different data.

## License and attribution

Weights: Apache 2.0 (as ModernBERT). Trained on data derived from CLINC150 (CC BY 3.0), Banking77 and MASSIVE
(CC BY 4.0), GoEmotions, Amazon Polarity and SciCite (Apache 2.0); see the dataset card for citations.

## Citation

```bibtex
@techreport{{moharana2026ctdreport,
  author      = {{Moharana, Lagnajit}},
  title       = {{Calibrated Typed Decisions: What a 7B Teacher's Soft Labels Do, and Don't, Teach a Small Classifier}},
  institution = {{Independent}},
  type        = {{Technical report}},
  year        = {{2026}},
  month       = nov,
  url         = {{{LINKS['report']}}}
}}
```
"""

for run in REPOS:
    with open(f"{EXPORT}/{run}/README.md", "w") as f:
        f.write(card(run))
print(card("F")[:2500])

---
license: apache-2.0
language:
- en
base_model: answerdotai/ModernBERT-base
datasets:
- m-lagnajit/calibrated-typed-decisions
pipeline_tag: text-classification
tags:
- calibration
- typed-decisions
- knowledge-distillation
- modernbert
---

# ctd-student-f: calibrated typed decisions, target F

A small (about 150M parameters) classifier that reads a **text, a question and a list of options** in one pass,
and returns a **calibrated chance for every option**: when it says 80%, it should be right about 80% of the time.
This repository holds **three students** trained with different seeds; their chances are averaged.

**Use it for **data like the six training sources**: short texts and a fixed set of options, such as intents, emotions, review sentiment or citation purposes.** It was chosen because it had the best familiar-data accuracy and calibration in the report, and was never reliably beaten there.
It is **not** reliable for new *kinds* of judgment: on PubMedQA, the best students in

## Cell 9 — Publish
Dry run first: lists what would be uploaded. Then set PUBLISH = True in Cell 2 and rerun Cells 2 and 9.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
for run, repo in REPOS.items():
    folder = f"{EXPORT}/{run}"
    files = sorted(os.listdir(folder))
    need = {"README.md", "config.json", "dials.json", "ctd_student.py"} | {f"{s}.safetensors" for s in SEEDS}
    assert need <= set(files), f"{run}: missing {need - set(files)}"
    print(f"{repo}:", ", ".join(f"{f} ({os.path.getsize(f'{folder}/{f}') / 1e6:.1f} MB)" for f in files))
    if PUBLISH:
        api.create_repo(repo, repo_type="model", exist_ok=True)
        api.upload_folder(folder_path=folder, repo_id=repo, repo_type="model",
                          commit_message="Calibrated typed-decision student: 3 seeds, dials, package, model card")
        print("  uploaded: https://huggingface.co/" + repo)
if not PUBLISH:
    print("\nDry run: nothing uploaded. Check the files and the card above, then set PUBLISH = True in Cell 2.")

m-lagnajit/ctd-student-f: README.md (0.0 MB), config.json (0.0 MB), ctd_student.py (0.0 MB), dials.json (0.0 MB), seed0.safetensors (298.0 MB), seed1.safetensors (298.0 MB), seed2.safetensors (298.0 MB)
  uploaded: https://huggingface.co/m-lagnajit/ctd-student-f
m-lagnajit/ctd-student-e: README.md (0.0 MB), config.json (0.0 MB), ctd_student.py (0.0 MB), dials.json (0.0 MB), seed0.safetensors (298.0 MB), seed1.safetensors (298.0 MB), seed2.safetensors (298.0 MB)
  uploaded: https://huggingface.co/m-lagnajit/ctd-student-e
